In [1]:
import pandas as pd
import numpy as np

from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)

In [2]:
from google.colab import files
train_df = files.upload()

Saving UNSW_NB15_training-set.parquet to UNSW_NB15_training-set.parquet


In [3]:
from google.colab import files
test_df = files.upload()

Saving UNSW_NB15_testing-set.parquet to UNSW_NB15_testing-set.parquet


In [4]:
train_df = pd.read_parquet("UNSW_NB15_training-set.parquet")
test_df = pd.read_parquet("UNSW_NB15_testing-set.parquet")

In [8]:
normal_train = train_df[train_df['label'] == 0]

print(normal_train.shape)

(56000, 36)


In [11]:
X_normal_train = normal_train.drop(columns=['label', 'attack_cat'])

print(X_normal_train.shape)

(56000, 34)


In [12]:
X_test = test_df.drop(columns=['label', 'attack_cat'])

y_test = test_df['label']

In [16]:
from sklearn.preprocessing import LabelEncoder
import pandas as pd

encoders = {}

# Re-initialize X_normal_train and X_test to ensure original categorical values
# This prevents issues if the cell is run multiple times
X_normal_train = normal_train.drop(columns=['label', 'attack_cat'])
X_test = test_df.drop(columns=['label', 'attack_cat'])

for col in ['proto', 'service', 'state']:

    le = LabelEncoder()

    # Fit on the combined unique values from both train_df and test_df
    # to ensure all possible categories are seen.
    combined_data = pd.concat([train_df[col], test_df[col]], axis=0).astype(str).unique()
    le.fit(combined_data)

    X_normal_train[col] = le.transform(
        X_normal_train[col].astype(str)
    )

    X_test[col] = le.transform(
        X_test[col].astype(str)
    )

    encoders[col] = le

In [17]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_normal_train_scaled = scaler.fit_transform(
    X_normal_train
)

X_test_scaled = scaler.transform(
    X_test
)

Isolation Forest Model Training

In [18]:
from sklearn.ensemble import IsolationForest

In [29]:
iso = IsolationForest(
    n_estimators=100,
    contamination=0.2,
    random_state=42,
    n_jobs=-1
)

iso.fit(X_normal_train_scaled)

IsolationForest(contamination=0.2, n_jobs=-1, random_state=42)

In [30]:
pred = iso.predict(X_test_scaled)

In [31]:
pred = np.where(
    pred == -1,
    1,
    0
)

In [32]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)

print(
    "Accuracy:",
    accuracy_score(y_test, pred)
)

print(
    "Precision:",
    precision_score(y_test, pred)
)

print(
    "Recall:",
    recall_score(y_test, pred)
)

print(
    "F1:",
    f1_score(y_test, pred)
)

print(
    confusion_matrix(y_test, pred)
)

print(
    classification_report(
        y_test,
        pred
    )
)

Accuracy: 0.585118787348783
Precision: 0.7006248204538925
Recall: 0.4304023647754346
F1: 0.5332331238043181
[[28663  8337]
 [25821 19511]]
              precision    recall  f1-score   support

           0       0.53      0.77      0.63     37000
           1       0.70      0.43      0.53     45332

    accuracy                           0.59     82332
   macro avg       0.61      0.60      0.58     82332
weighted avg       0.62      0.59      0.58     82332

